使用transformers库来调用Qwen大模型，我们先要安装好transformers、pytorch、tokenizers等库。推荐最新版本。

参考文档：
- https://huggingface.co/Qwen/Qwen2-7B-Instruct

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig
from IPython.display import Markdown

### 1. 准备模型和分词器

In [2]:
model_name = "Qwen/Qwen2-7B-Instruct"

In [3]:
# 第一次下载模型会比较慢，3.95G * 4
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # "../../models/Qwen/Qwen2-7B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# 分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


当缓存中不存在模型的时候，每次实例化的时候，都需要从hugging face下载。     
**为了防止缓存被清理了，我们可以保存模型和分词器到某个目录中。**

In [5]:
model_tokenizer_dir = "../../models/Qwen/Qwen2-7B-Instruct"
# 查看目录中的文件
!ls {model_tokenizer_dir}

added_tokens.json                model-00004-of-00004.safetensors
config.json                      model.safetensors.index.json
generation_config.json           special_tokens_map.json
merges.txt                       tokenizer.json
model-00001-of-00004.safetensors tokenizer_config.json
model-00002-of-00004.safetensors vocab.json
model-00003-of-00004.safetensors


In [6]:
# 保存模型和分词器
# model.save_pretrained(model_tokenizer_dir)
# tokenizer.save_pretrained(model_tokenizer_dir)

# 再次查看目录中的文件
# !ls {model_tokenizer_idr}

### 2. 准备输入文本

In [7]:
input_text = "Pytorch是什么，它主要用于什么方面？"
tokens = tokenizer(input_text, return_tensors="pt")

In [8]:
tokens

{'input_ids': tensor([[ 13828,  27414, 102021,   3837,  99652, 113078,  99245,  99522,  11319]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

### 3. 模型进行推理

In [9]:
# 根据自己需要，调整相关参数
config = GenerationConfig(max_length=1000, temperature=0.7, do_sample=True)
config

GenerationConfig {
  "do_sample": true,
  "max_length": 1000,
  "temperature": 0.7
}

In [18]:
# 使用GPU性能会显著提升
model.to("mps")
# 输出出耗时时间
%time output_ids = model.generate(tokens.input_ids.to("mps"), attention_mask=tokens.attention_mask.to("mps"), max_length=1000, temperature=0.8)
output_ids.shape

CPU times: user 15.5 s, sys: 1.53 s, total: 17.1 s
Wall time: 21.6 s


torch.Size([1, 336])

体验了一下，没有`ollama run qwen2:7b-instruct`然后直接提问效率快。

In [11]:
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
print(output_text)

Pytorch是什么，它主要用于什么方面？ PyTorch是一个开源的深度学习框架，由Facebook的AI研究部门（现称为FAIR）创建，并在2017年开源发布。它基于Python语言开发，旨在提供灵活和强大的计算图以支持高效的机器学习模型训练与推理。

PyTorch的主要特点包括：

1. **动态计算图**：PyTorch支持动态构建计算图，这意味着开发者可以在运行时修改网络结构或输入数据的大小，这使得模型开发和实验更加灵活。

2. **自动求导**：PyTorch内置了自动微分功能，可以轻松地计算模型参数的梯度，简化了梯度计算过程，有利于快速进行模型优化。

3. **GPU加速**：PyTorch充分利用了GPU硬件资源，通过CUDA库加速模型的训练和推理过程，显著提高计算性能。

4. **社区活跃**：PyTorch拥有一个庞大的开发者社区，提供了丰富的教程、预训练模型以及各种工具包，便于用户快速上手并进行创新应用。

5. **易于使用**：与TensorFlow等其他深度学习框架相比，PyTorch的API设计更为直观，易于理解和使用，尤其是对于初学者而言。

PyTorch广泛应用于以下领域：

- **自然语言处理（NLP）**：如文本生成、情感分析、问答系统等。
- **计算机视觉**：图像分类、目标检测、图像分割、风格迁移等。
- **语音识别**：语音合成、语音识别、对话系统等。
- **强化学习**：构建和训练复杂的决策模型，用于游戏、机器人控制等领域。
- **推荐系统**：个性化内容推荐、协同过滤等。
- **生物信息学**：蛋白质结构预测、基因序列分析等。

总之，PyTorch是一个功能强大且易于使用的深度学习框架，适用于多种应用场景，特别适合需要灵活性和快速迭代的项目。无论是科研还是工业应用，PyTorch都是一个值得考虑的选择。<|im_end|>


In [12]:
# output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
# print(output_text)
# 加了skip_special_tokens=True,会去掉`<|im_end|>`这种特殊的字符

In [13]:
len(output_text)

810

In [14]:
Markdown(output_text)

Pytorch是什么，它主要用于什么方面？ PyTorch是一个开源的深度学习框架，由Facebook的AI研究部门（现称为FAIR）创建，并在2017年开源发布。它基于Python语言开发，旨在提供灵活和强大的计算图以支持高效的机器学习模型训练与推理。

PyTorch的主要特点包括：

1. **动态计算图**：PyTorch支持动态构建计算图，这意味着开发者可以在运行时修改网络结构或输入数据的大小，这使得模型开发和实验更加灵活。

2. **自动求导**：PyTorch内置了自动微分功能，可以轻松地计算模型参数的梯度，简化了梯度计算过程，有利于快速进行模型优化。

3. **GPU加速**：PyTorch充分利用了GPU硬件资源，通过CUDA库加速模型的训练和推理过程，显著提高计算性能。

4. **社区活跃**：PyTorch拥有一个庞大的开发者社区，提供了丰富的教程、预训练模型以及各种工具包，便于用户快速上手并进行创新应用。

5. **易于使用**：与TensorFlow等其他深度学习框架相比，PyTorch的API设计更为直观，易于理解和使用，尤其是对于初学者而言。

PyTorch广泛应用于以下领域：

- **自然语言处理（NLP）**：如文本生成、情感分析、问答系统等。
- **计算机视觉**：图像分类、目标检测、图像分割、风格迁移等。
- **语音识别**：语音合成、语音识别、对话系统等。
- **强化学习**：构建和训练复杂的决策模型，用于游戏、机器人控制等领域。
- **推荐系统**：个性化内容推荐、协同过滤等。
- **生物信息学**：蛋白质结构预测、基因序列分析等。

总之，PyTorch是一个功能强大且易于使用的深度学习框架，适用于多种应用场景，特别适合需要灵活性和快速迭代的项目。无论是科研还是工业应用，PyTorch都是一个值得考虑的选择。<|im_end|>